# MobileFaceNet

## Add New Embeddings

In [2]:
!pip3 install torch torchvision torchaudio

In [ ]:
!pip3 install scikit-learn


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import os
import sklearn
from PIL import Image
from sklearn.neighbors import KNeighborsClassifier


In [5]:


# cd to /content/drive/MyDrive/Face_Dataset
%cd Face_Dataset

print(os.getcwd())

g:\.thesis\named-ai\data-preprocessing\Face_Dataset
g:\.thesis\named-ai\data-preprocessing\Face_Dataset


In [5]:
# Step 1 Clone the MobileFaceNet repository
!git clone https://github.com/foamliu/MobileFaceNet.git

fatal: destination path 'MobileFaceNet' already exists and is not an empty directory.


In [11]:
%cd ..

g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet


In [6]:
print("Current working directory:", os.getcwd())

# Step 2: Change directory into the repo
%cd ..

# Step 3: Make a directory for weights
!mkdir -p weights

# Step 4: Change into weights directory
%cd weights

# Step 5: Download the pretrained model
#!wget https://github.com/foamliu/MobileFaceNet/releases/download/v1.0/mobilefacenet.pt

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset
g:\.thesis\named-ai\data-preprocessing
g:\.thesis\named-ai\data-preprocessing\weights


In [14]:
%cd MobileFaceNet

g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet


In [15]:
from mobilefacenet import MobileFaceNet
import torch
import time

# Load model
filename = 'weights/mobilefacenet.pt'
print(f'Loading {filename}...')
start = time.time()
model = MobileFaceNet()
model.load_state_dict(torch.load(filename, map_location=torch.device('cpu')))
model.eval()
print('Elapsed {:.2f} sec'.format(time.time() - start))


Loading weights/mobilefacenet.pt...
Elapsed 0.16 sec


In [106]:
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

def get_embedding(image_path):
    img = Image.open(image_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0)        # (1,3,112,112)

    with torch.no_grad():
        emb = model(input_tensor)

        # ---- flatten to 1-D ----
        if isinstance(emb, (tuple, list)):
            emb = emb[0]                              # some nets return tuple
        emb = emb.squeeze()

        # ---- force it to CPU NumPy ----
        if isinstance(emb, torch.Tensor):
            emb = emb.detach().cpu().numpy()

    # ---- L2 normalise ----
    norm = np.linalg.norm(emb)
    if norm > 0:
        emb = emb / norm
    return emb.astype(np.float32)

In [82]:
def buildDB(facebank_dir = 'facebank'):
    """
    Scans a directory structure like:
        facebank/PersonName/*.jpg
    and builds a dict {person_name: [embeddings]}.

    Returns:
        face_db: dict of {str: [np.ndarray]}
    """
    face_db = {}
    for person_name in os.listdir(facebank_dir):
        person_path = os.path.join(facebank_dir, person_name)
        if not os.path.isdir(person_path): 
            continue
        embs = []
        for f in os.listdir(person_path):
            if not f.lower().endswith(('.jpg','.jpeg','.png')): 
                continue
            emb = get_embedding(os.path.join(person_path, f))
            # make sure it's numpy and normalized
            embs.append(np.asarray(emb, dtype=np.float32))
        if embs:
            face_db[person_name] = embs
    return face_db

In [20]:
cd ..

g:\.thesis\named-ai\data-preprocessing


In [83]:
def train_knn(face_db, n_neighbors=1, metric ='euclidean'):
    """
    Trains KNN using averaged embeddings per person.
    """
    X, y = [], []
    for name, embs in face_db.items():
        arr = np.stack(embs)                   # shape (n_images, 128)
        mean_emb = arr.mean(axis=0)
        mean_emb = mean_emb / np.linalg.norm(mean_emb)   # normalize again
        X.append(mean_emb.astype(np.float32))
        y.append(name)
    X = np.stack(X)
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric=metric)
    knn.fit(X, y)
    return knn


In [ ]:
def recognize_face_knn(image_path, knn, threshold=0.9, debug=False):
    """
    Recognize a face using a trained KNN model and threshold.

    Args:
        image_path (str): Path to the face image to recognize.
        knn (KNeighborsClassifier): Trained KNN classifier.
        threshold (float): Distance threshold for rejecting unknowns.
                           For metric='euclidean', try 0.8–1.2
                           For metric='cosine', try 0.3–0.5
        debug (bool): Print extra details about the match.

    Returns:
        (name, distance): tuple(str, float)
    """
    emb = get_embedding(image_path)
    dist, idx = knn.kneighbors([emb], n_neighbors=1)
    name = knn.predict([emb])[0]
    distance = float(dist[0][0])

    if debug:
        print(f"[DEBUG] Predicted: {name}, Distance: {distance:.4f}")

    if distance > threshold:
        return "Unknown", distance
    return name, distance


In [46]:
import joblib

def save_db(knn, filename = 'facebank_mfn.pkl'):
    joblib.dump(knn, filename)
    print(f"[INFO] KNN model saved to {filename}")
def load_db(filename = 'facebank_mfn.pkl'):
    knn = joblib.load(filename)
    print(f"[INFO] Loaded KNN model from {filename}")
    return knn

In [100]:
face_db = buildDB()
knn = train_knn(face_db, n_neighbors=3)
save_db(knn)

[INFO] KNN model saved to facebank_mfn.pkl


In [101]:
for name, embs in face_db.items():
    print(name, type(embs[0]), embs[0].shape, embs[0].dtype)

Akshay Kumar <class 'numpy.ndarray'> (128,) float32
Alexandra Daddario <class 'numpy.ndarray'> (128,) float32
Alia Bhatt <class 'numpy.ndarray'> (128,) float32
Amitabh Bachchan <class 'numpy.ndarray'> (128,) float32
Andy Samberg <class 'numpy.ndarray'> (128,) float32
Anushka Sharma <class 'numpy.ndarray'> (128,) float32
Billie Eilish <class 'numpy.ndarray'> (128,) float32
Brad Pitt <class 'numpy.ndarray'> (128,) float32
Camila Cabello <class 'numpy.ndarray'> (128,) float32
Charlize Theron <class 'numpy.ndarray'> (128,) float32
Claire Holt <class 'numpy.ndarray'> (128,) float32
Courtney Cox <class 'numpy.ndarray'> (128,) float32
Dwayne Johnson <class 'numpy.ndarray'> (128,) float32
Elizabeth Olsen <class 'numpy.ndarray'> (128,) float32
Ellen Degeneres <class 'numpy.ndarray'> (128,) float32
Henry Cavill <class 'numpy.ndarray'> (128,) float32
Hrithik Roshan <class 'numpy.ndarray'> (128,) float32
Hugh Jackman <class 'numpy.ndarray'> (128,) float32
Jessica Alba <class 'numpy.ndarray'> (128,

In [1]:
print("Current working directory:", os.getcwd())
e = get_embedding('test_images/Akshay Kumar_0.jpg')
print(type(e), e.shape, e.dtype)




NameError: name 'os' is not defined

In [103]:
#load knn db from file
knn = load_db('facebank_mfn.pkl')

[INFO] Loaded KNN model from facebank_mfn.pkl


In [109]:
def debug_embedding(image_path, knn):
    emb = get_embedding(image_path)
    print("type:", type(emb), "dtype:", emb.dtype, "shape:", emb.shape)
    print("first 5 values:", emb[:5])
    dists, idx = knn.kneighbors([emb], n_neighbors=1)
    print("kneighbors distance:", dists[0][0])
    return dists[0][0]

In [110]:
print(debug_embedding('test_images/Akshay Kumar_0.jpg', knn))
print(debug_embedding('test_images/Akshay Kumar_47.jpg', knn))

type: <class 'numpy.ndarray'> dtype: float32 shape: (128,)
first 5 values: [ 0.00938351  0.00767279  0.08549164 -0.1265889   0.0562075 ]
kneighbors distance: 0.6079503297805786
0.6079503297805786
type: <class 'numpy.ndarray'> dtype: float32 shape: (128,)
first 5 values: [-0.021633   -0.0978827  -0.0561024  -0.05735286  0.10989715]
kneighbors distance: 0.5546225309371948
0.5546225309371948


In [126]:
# Example (using euclidean metric)
result, dist = recognize_face_knn('test_images/Akshay Kumar_0.jpg', knn, threshold=0.9, debug=False)
print("Prediction:", result, "| Distance:", dist)


Prediction: Billie Eilish | Distance: 0.6079503297805786


In [15]:
# test_embedding = get_embedding('my_images/kyle_156.jpg')
# test_embedding = get_embedding('../processed_whole_face_dataset/val/kyle/kyle_161.jpg')
# test_embedding = get_embedding('../processed_whole_face_dataset/val/Zac Efron/Zac Efron_2.jpg')

test_embedding = get_embedding('../processed_whole_face_dataset/val/Tom Cruise/Tom Cruise_17.jpg')
print(test_embedding)

tensor([-8.9243e-02,  1.2614e+00, -4.9276e-01,  1.9606e+00,  1.0549e+00,
         1.3994e+00, -2.0412e+00,  1.1254e+00,  1.2575e+00,  6.8627e-01,
         8.5237e-02,  2.1320e+00,  5.3142e-01,  1.9682e+00, -2.5576e+00,
         8.5788e-01,  6.0927e-01, -4.9489e-01,  1.3770e+00,  1.0110e+00,
        -1.0470e+00,  4.2657e+00,  2.0887e+00,  1.6598e+00, -2.3915e-01,
         2.7591e+00, -1.1048e-03, -1.6891e+00,  6.5124e-01, -8.4227e-01,
         1.8632e+00,  2.4158e+00, -1.3980e+00, -2.6148e-01, -1.1415e+00,
         1.4403e+00,  5.3336e-02,  1.2166e+00, -4.3740e-01,  1.5985e+00,
         3.4416e-01,  1.5826e+00, -1.8154e+00,  3.4685e-01,  7.0870e-01,
        -6.9588e-01, -9.5989e-01, -6.4239e-01,  1.7788e+00, -4.6110e+00,
         1.1667e+00, -1.9093e+00,  1.8574e+00, -3.2795e+00, -3.5931e+00,
        -2.0152e-01,  2.9221e-01, -1.1172e+00,  2.0644e+00, -4.6767e-01,
         1.4042e+00,  1.3651e+00,  2.7450e-01, -1.6604e+00,  1.4132e-01,
        -6.9465e-01,  1.0509e+00,  4.1445e-02,  3.4

In [56]:
from torch.nn.functional import cosine_similarity

def recognize_face(test_embedding, face_db, threshold=0.6):
    max_sim = 0
    identity = "Unknown"
    for name, db_embedding in face_db.items():
        sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
        if sim.item() > max_sim and sim.item() > threshold:
            max_sim = sim.item()
            identity = name
    return identity, max_sim


In [ ]:
def recognize_face(image, face_db, threshold=0.5):
    """
    Args:
        image: input image for recognition
        face_db: dict {name: embedding}
        threshold: minimum similarity to accept a match

    Returns:
        (best_match_name, best_similarity)
    """
    if not face_db:
        print("[WARN] face_db is empty.")
        return "Unknown", 0.0

    target_emb = get_embedding(image)
    best_match = "Unknown"
    best_score = -1

    for name, db_emb in face_db.items():
        # Compute cosine similarity
        sim = np.dot(target_emb, db_emb) / (np.linalg.norm(target_emb) * np.linalg.norm(db_emb))
        
        if sim > best_score:
            best_score = sim
            best_match = name

    if best_score < threshold:
        best_match = "Unknown"

    return best_match, best_score

In [21]:
print("Current working directory:", os.getcwd())
%cd ..

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset
g:\.thesis\named-ai\data-preprocessing


In [36]:
# recognize face

name, score = recognize_face('my_images/kyle_156.jpg', face_db)
print(f"Recognized: {name} (Similarity: {score:.2f})")

Recognized: kyle (Similarity: 0.71)


C:\Users\julia\AppData\Local\Temp\ipykernel_13284\3388533542.py:24: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  sim = np.dot(target_emb, db_emb) / (np.linalg.norm(target_emb) * np.linalg.norm(db_emb))
